Import libaries and functions

In [1]:
import numpy as np
import torch
import pyro
from torch import nn
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

from codebase_0 import set_seed, DeviceLoader
from gamma_bcann_vi import (
    GammaBCANN, gamma_fit_glm_mle, gamma_build_bcann_guide, 
    gamma_bcann_svi_training_loop, gamma_weighted_mean
)

from gamma_bcann_hmc_nuts import (
    gamma_stratified_subsample, gamma_make_chain_inits, gamma_run_nuts, 
    gamma_time_nuts
)
from gamma_horseshoe import (
    HorseshoeGammaBCANN, gamma_build_guide_horseshoe_bcann
)

from gamma_nuts_parallel import gamma_run_nuts_parallel

seed = 42
set_seed(seed)
pyro.clear_param_store()

Load data and perform train/validation/test split

In [2]:
device = torch.device("cuda:2")

torch.set_default_device("cpu")

dataset = pd.read_csv("freMTPL2sev_transformed.csv", index_col= 0) 
y = torch.tensor(dataset["ClaimAmount"].to_numpy(), dtype=torch.float32)
weights = torch.ones(len(y)) 
X = torch.tensor(
    dataset.drop(columns=["IDpol", "Exposure","ClaimAmount"]).to_numpy(), 
    dtype=torch.float32
)   # 48 columns

# Learning-test set split
X_learn, X_test, y_learn, y_test, w_learn, w_test = train_test_split(
    X, y, weights, test_size= 0.1, random_state= seed
)

# Training-CV set split
X_train, X_val, y_train, y_val, w_train, w_val = train_test_split(
    X_learn, y_learn, w_learn, test_size= 0.1, random_state= seed
)

learn_scaler = MinMaxScaler().fit(X_learn)
X_learn = learn_scaler.transform(X_learn)
X_test = learn_scaler.transform(X_test)

train_scaler = MinMaxScaler().fit(X_train)
X_train = train_scaler.transform(X_train)
X_val = train_scaler.transform(X_val)

def T(a):
    return torch.as_tensor(a, dtype=torch.float32, device=device)

X_train, X_val, X_learn, X_test = map(T, (X_train, X_val, X_learn, X_test))
y_train, y_val, y_learn, y_test = map(T, (y_train, y_val, y_learn, y_test))
w_train, w_val, w_learn, w_test = map(T, (w_train, w_val, w_learn, w_test))

torch.set_default_device(device)
train_loader = DeviceLoader(
        X_train, y_train, w_train,
        batch_size = 2056,
        shuffle= True
)
val_loader = DeviceLoader(
        X_val, y_val, w_val,
        batch_size= 10000
)
learn_loader = DeviceLoader(
        X_learn, y_learn, w_learn,
        batch_size= 10000
)

Initialise HorseshoeGammaBCANN, VI posterior (as guide) and create a chain object for NUTS

In [ ]:
network_architecture = [20,15, 10]
y_scale = gamma_weighted_mean(y_train, w_train)
glm_mle = gamma_fit_glm_mle(X_learn, y_learn, w_learn, random_state=seed)  

with torch.device(device):
    horseshoe_gamma_bcann = HorseshoeGammaBCANN(
        in_features= 48 ,
        neural_net_params = {"hidden_activation_nodes": network_architecture},
        y_scale=y_scale,
    )                                           # no .to(device)
    glm_mle = tuple(t.to(device) for t in glm_mle)
    guide = gamma_build_guide_horseshoe_bcann(
        horseshoe_gamma_bcann,
        X_train[:256], y_train[:256],
        glm_mle=glm_mle
    )
inits = gamma_make_chain_inits(
    horseshoe_gamma_bcann, 
    X_learn, 
    y_learn, 
    w_learn, 
    num_chains=6, 
    glm_mle=glm_mle,
    guide=guide, 
    how="svi_sample", 
    seed=seed, 
    device = device
)

NUTS test run see how long it takes per iteration

In [5]:
gamma_time_nuts(
    horseshoe_gamma_bcann, 
    X_learn, 
    y_learn, 
    w_learn,
    inits,
    n_iter = 50, 
    device = device, 
    jit_compile= True
)

2.04 s / iteration on cuda:2  (early warmup iterations are not representative of tree depth)


2.037075538635254

Full NUTS run

In [9]:
WARMUP, SAMPLES = 1500, 1500
TARGET_ACCEPT = 0.9
MAX_TREE_DEPTH = 8
gamma_horseshoe_samples, gamma_horseshoe_info = gamma_run_nuts_parallel(
    dict(
        in_features=48, 
        neural_net_params={
            "hidden_activation_nodes": network_architecture,
            "hidden_activations": nn.Tanh
        }
    ),   # GammaBCANN(**model_kwargs)
    X_learn, y_learn, w_learn, inits, 
    out_dir="gamma_nuts_chains_horseshoe",
    model_cls = HorseshoeGammaBCANN,
    num_warmup=WARMUP, num_samples=SAMPLES, 
    target_accept=TARGET_ACCEPT, 
    max_tree_depth=MAX_TREE_DEPTH,
    seed=seed, 
    device=device, 
    progress_every= 5,
    jit_compile = True
)

elapsed 4:32:24
chain 5: Sample: 100%|██████████| 3000/3000 [4:27:21,  5.35s/it, step size=2.73e-03, acc. prob=0.881]


Save to disk for later uses

In [10]:
torch.save(
    {
        "samples": gamma_horseshoe_samples,
        "info": gamma_horseshoe_info,
        "config": dict(num_warmup=WARMUP, num_samples=SAMPLES, target_accept=TARGET_ACCEPT,
                       max_tree_depth=MAX_TREE_DEPTH, seed=seed, n_chains=6),
    },
    "hmc_results/gamma_horseshoe_nuts_full_set.pt",
)
